In [1]:
from pathlib import Path
import sys

def find_repo_root(start=None):
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "config" / "paths.json").exists():
            return candidate
    raise FileNotFoundError("Could not locate repo root containing config/paths.json")

REPO_ROOT = find_repo_root()
UTILS_DIR = REPO_ROOT / "code" / "utils"
if str(UTILS_DIR) not in sys.path:
    sys.path.append(str(UTILS_DIR))

from project_paths import ensure_project_dirs, load_project_paths

PATHS = ensure_project_dirs(load_project_paths(REPO_ROOT))

import os
import pandas as pd
from pathlib import Path


### Definitions

In [2]:

bids_dir = PATHS["bids_root"]
petpre_hmc_dir = bids_dir / 'derivatives' / 'petprep_hmc'

participants_file = PATHS["participants_file"]
participants_df = pd.read_csv(participants_file, delimiter='\t')

group_allocation = {
    'drug': {'ses': 'ses-01', 'drug_allocation': 1},
    'placebo': {'ses': 'ses-02', 'drug_allocation': 1}
}

colors = {'Placebo': '#3C3C3C', 'DMT + har': '#E69F00'}


In [3]:
import pandas as pd

# List to collect individual dataframes
motion_df_list = []

# Iterate through subjects and sessions
for _, row in participants_df.iterrows():
    subject = row["participant_id"]
    
    for session, condition in [("ses-01", row["drug_ses-01"]), ("ses-02", row["drug_ses-02"])]:
        
        motion_file = petpre_hmc_dir / subject / session / f"{subject}_{session}_desc-confounds_timeseries.tsv"
        
        if motion_file.exists():
            df = pd.read_csv(motion_file, sep="\t")
            df["participant_id"] = subject
            df["session"] = session
            df["drug_allocation"] = condition
            motion_df_list.append(df)
        else:
            print(f"Missing file: {motion_file}")

# Combine into one dataframe
motion_df = pd.concat(motion_df_list, ignore_index=True)

# Extract max 'max_tot' per subject and session
max_motion_df = (
    motion_df.groupby(["participant_id", "session", "drug_allocation"])["max_tot"]
    .max()
    .reset_index()
)


In [4]:
motion_df

,Unnamed: 0,trans_x,trans_y,trans_z,rot_x,rot_y,rot_z,max_x,max_y,max_z,max_tot,median_tot,participant_id,session,drug_allocation
0,0,-0.352683,0.023287,-0.975653,-0.000692,-0.000415,0.000084,0.349659,0.095145,1.042091,1.098630,0.992246,sub-HDP01,ses-01,Placebo
1,1,-0.352683,0.023287,-0.975653,-0.000692,-0.000415,0.000084,0.349659,0.095145,1.042091,1.098630,0.992246,sub-HDP01,ses-01,Placebo
2,2,-0.352683,0.023287,-0.975653,-0.000692,-0.000415,0.000084,0.349659,0.095145,1.042091,1.098630,0.992246,sub-HDP01,ses-01,Placebo
3,3,-0.352683,0.023287,-0.975653,-0.000692,-0.000415,0.000084,0.349659,0.095145,1.042091,1.098630,0.992246,sub-HDP01,ses-01,Placebo
4,4,-0.352683,0.023287,-0.975653,-0.000692,-0.000415,0.000084,0.349659,0.095145,1.042091,1.098630,0.992246,sub-HDP01,ses-01,Placebo
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
639,18,3.747298,-10.597826,11.010367,0.062380,0.014853,-0.017044,2.642739,9.173056,9.041387,12.510718,5.720328,sub-HDP19,ses-02,Verum
640,19,1.037983,-2.543630,-0.919830,0.007855,0.008938,-0.010045,2.008535,1.512390,1.724530,2.653516,1.718587,sub-HDP19,ses-02,Verum
641,20,0.219661,3.751898,-9.836711,-0.029862,0.010398,-0.003860,1.640311,4.253202,6.454123,7.688605,4.548475,sub-HDP19,ses-02,Verum
642,21,-2.313974,-0.788731,4.105064,0.024264,0.006845,0.007700,2.139852,2.003995,2.869480,3.636018,2.138566,sub-HDP19,ses-02,Verum


In [5]:
from scipy.stats import ttest_rel

# Pivot to align each subject's values by drug condition
pivot_df = max_motion_df.pivot(index="participant_id", columns="drug_allocation", values="max_tot")

# Drop any subjects with missing data
pivot_df = pivot_df.dropna(subset=["Placebo", "Verum"])

# Paired t-test
t_stat, p_val = ttest_rel(pivot_df["Verum"], pivot_df["Placebo"])

# Calculate difference per subject for inspection (optional)
pivot_df["diff_DMT_placebo"] = pivot_df["Verum"] - pivot_df["Placebo"]

# Print results
print("Mean motion (Placebo):", pivot_df["Placebo"].mean())
print("Mean motion (DMT + har):", pivot_df["Verum"].mean())
print("Mean difference (DMT - Placebo):", pivot_df["diff_DMT_placebo"].mean())
print(f"Paired t-test: t = {t_stat:.3f}, p = {p_val:.4f}")

Mean motion (Placebo): 6.853361968576338
Mean motion (DMT + har): 8.065244594456805
Mean difference (DMT - Placebo): 1.2118826258804691
Paired t-test: t = 0.763, p = 0.4589
